# Chess RL Engine — Kaggle Training

**Kaggle GPU tips applied here:**
- Only turn on GPU once you're ready to run — don't leave it idle
- Download the .ipynb file to save progress, do NOT use the Commit button
- Stop the session manually before closing the browser tab (Active Events → Stop)
- Run sanity check locally before starting the GPU session

**Setup (do this before turning GPU on):**
1. Upload `chess-rl-engine.zip` as a Kaggle Dataset (+New Dataset → upload zip)
2. Add it: Notebook settings → Data → Add your dataset
3. Only then switch Accelerator to **GPU P100**

**24-hour GPU plan:**
- Stage 1 (~4 hrs): pop 8, 15 gen — warmup
- Stage 2 (~10 hrs): pop 16, 25 gen — main training
- Stage 3 (~10 hrs): pop 20, 30 gen — polish

Expected final ELO: **1550–1700** (+ ~200-300 more with MCTS at inference)

In [ ]:
# ── Cell 1: Setup (CPU only, run before enabling GPU) ─────────────────────
# This cell does not use the GPU. Run it with CPU accelerator to verify
# your dataset is mounted correctly before you switch to GPU.

import os, zipfile, subprocess, sys

ZIP_PATH = '/kaggle/input/chess-rl-engine/chess-rl-engine.zip'  # update if your dataset name differs
WORK_DIR = '/kaggle/working'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(WORK_DIR)

PROJECT_DIR = os.path.join(WORK_DIR, 'chess-rl-engine')
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

print('Project extracted to:', PROJECT_DIR)
print('Contents:', os.listdir(PROJECT_DIR))

In [ ]:
# ── Cell 2: Install dependencies + confirm GPU ────────────────────────────
# Switch to GPU P100 BEFORE running this cell.
# PyTorch is pre-installed on Kaggle — only chess and tqdm needed.

subprocess.run(['pip', 'install', 'python-chess', 'tqdm', '-q'], check=True)

import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Switch accelerator to GPU P100 in notebook settings.')

In [ ]:
# ── Cell 3: STAGE 1 — Warmup (~4 hours) ──────────────────────────────────
# Small population, shorter games. Gets agents past the random-play phase.
# Expected ELO at end: 1300-1380
#
# When complete: File → Download Notebook to save your progress.

from utils.config import Config, GeneticConfig, PPOConfig, TrainingConfig, GameConfig
from training.evolution import EvolutionLoop

stage1_cfg = Config(
    genetic=GeneticConfig(
        population_size=8,
        num_generations=15,
        games_per_evaluation=8,
        mutation_rate=0.20,     # high early = more exploration
    ),
    ppo=PPOConfig(
        games_per_update=16,
        learning_rate=5e-4,
    ),
    game=GameConfig(
        max_moves=150,          # shorter games = faster iterations early
    ),
    training=TrainingConfig(
        checkpoint_dir='/kaggle/working/checkpoints/stage1',
        log_dir='/kaggle/working/logs/stage1',
        checkpoint_every=5,
    ),
)

print('STAGE 1: Warmup | 15 gen | pop 8 | ~4 hours')
stage1_loop = EvolutionLoop(stage1_cfg)
stage1_loop.run()

best_s1 = stage1_loop.population.best_agent()
best_s1.save('/kaggle/working/checkpoints/stage1_best.pt')
print(f'Stage 1 done. Best ELO: {best_s1.elo:.0f}')
print('ACTION: File → Download Notebook now to checkpoint your progress.')

In [ ]:
# ── Cell 4: STAGE 2 — Main training (~10 hours) ───────────────────────────
# Larger population, full-length games. Most learning happens here.
# Expected ELO at end: 1430-1550

stage2_cfg = Config(
    genetic=GeneticConfig(
        population_size=16,
        num_generations=25,
        games_per_evaluation=10,
        mutation_rate=0.15,
    ),
    ppo=PPOConfig(
        games_per_update=20,
        learning_rate=3e-4,
    ),
    game=GameConfig(
        max_moves=200,
    ),
    training=TrainingConfig(
        checkpoint_dir='/kaggle/working/checkpoints/stage2',
        log_dir='/kaggle/working/logs/stage2',
        checkpoint_every=5,
    ),
)

print('STAGE 2: Main training | 25 gen | pop 16 | ~10 hours')
stage2_loop = EvolutionLoop(stage2_cfg)
stage2_loop.population.initialise()

# Seed the best agent from Stage 1 into Stage 2's population
s1_path = '/kaggle/working/checkpoints/stage1_best.pt'
if os.path.exists(s1_path):
    stage2_loop.population.agents[0].load(s1_path)
    print('Seeded Stage 1 best agent into Stage 2 population.')

stage2_loop.run()

best_s2 = stage2_loop.population.best_agent()
best_s2.save('/kaggle/working/checkpoints/stage2_best.pt')
print(f'Stage 2 done. Best ELO: {best_s2.elo:.0f}')
print('ACTION: File → Download Notebook now to checkpoint your progress.')

In [ ]:
# ── Cell 5: STAGE 3 — Polish (~10 hours) ──────────────────────────────────
# Lower LR + lower mutation = exploit what we've found, don't explore randomly.
# Expected ELO at end: 1550-1700

stage3_cfg = Config(
    genetic=GeneticConfig(
        population_size=20,
        num_generations=30,
        games_per_evaluation=12,
        mutation_rate=0.10,
        mutation_strength=0.10,
    ),
    ppo=PPOConfig(
        games_per_update=20,
        learning_rate=1e-4,     # fine-tune with lower LR
    ),
    game=GameConfig(
        max_moves=200,
    ),
    training=TrainingConfig(
        checkpoint_dir='/kaggle/working/checkpoints/stage3',
        log_dir='/kaggle/working/logs/stage3',
        checkpoint_every=5,
    ),
)

print('STAGE 3: Polish | 30 gen | pop 20 | ~10 hours')
stage3_loop = EvolutionLoop(stage3_cfg)
stage3_loop.population.initialise()

s2_path = '/kaggle/working/checkpoints/stage2_best.pt'
if os.path.exists(s2_path):
    stage3_loop.population.agents[0].load(s2_path)
    print('Seeded Stage 2 best agent into Stage 3 population.')

stage3_loop.run()

best_s3 = stage3_loop.population.best_agent()
best_s3.save('/kaggle/working/checkpoints/FINAL_best_agent.pt')
print(f'Stage 3 done. Best ELO: {best_s3.elo:.0f}')
print('Final agent saved: /kaggle/working/checkpoints/FINAL_best_agent.pt')

In [ ]:
# ── Cell 6: Generate outputs (curves + GIF) ───────────────────────────────
# Run this after all 3 stages complete.
# Produces training_curves.png and game.gif for your GitHub repo.

import json, torch
import matplotlib.pyplot as plt
import chess, chess.svg
from rl_agent.agent import ChessAgent
from rl_agent.mcts import MCTS

subprocess.run(['pip', 'install', 'cairosvg', 'Pillow', '-q'], check=True)
import cairosvg
from PIL import Image
import io

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Training curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chess RL Engine — 24hr Training Run', fontsize=14, fontweight='bold')
stages = [
    ('/kaggle/working/logs/stage1', 'Stage 1 (warmup)',  '#3498db'),
    ('/kaggle/working/logs/stage2', 'Stage 2 (main)',    '#2ecc71'),
    ('/kaggle/working/logs/stage3', 'Stage 3 (polish)',  '#e74c3c'),
]
gen_offset = 0
for log_dir, label, color in stages:
    for fname in ['metrics_final.json', 'metrics.json']:
        path = os.path.join(log_dir, fname)
        if os.path.exists(path):
            with open(path) as f:
                m = json.load(f)
            gens = [g + gen_offset for g in m['generation']]
            gen_offset = gens[-1]
            axes[0].plot(gens, m['best_elo'],  color=color, label=label, linewidth=2)
            axes[1].plot(gens, m['win_rate'],  color=color, label=label, linewidth=2)
            break

axes[0].axhline(y=1200, color='gray', linestyle=':', alpha=0.5, label='Start ELO')
axes[0].set(title='ELO Progression', xlabel='Generation', ylabel='ELO')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set(title='Win Rate', xlabel='Generation', ylabel='Win Rate', ylim=(0,1))
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
print('Saved: training_curves.png')
plt.show()

# ── Game GIF ──────────────────────────────────────────────────────────────
agent = ChessAgent(device=device)
agent.load('/kaggle/working/checkpoints/FINAL_best_agent.pt')
print(f'Agent loaded. ELO: {agent.elo:.0f}')

mcts = MCTS(agent.network, num_simulations=100, device=device)
board = chess.Board()
frames = []

def board_to_frame(b, last_move=None):
    svg = chess.svg.board(b, lastmove=last_move, size=400)
    png = cairosvg.svg2png(bytestring=svg.encode())
    return Image.open(io.BytesIO(png)).convert('RGB')

frames.append(board_to_frame(board))
for _ in range(80):
    if board.is_game_over(): break
    move = mcts.select_action(board, temperature=0.3)
    board.push(move)
    frames.append(board_to_frame(board, move))

frames.extend([frames[-1]] * 6)  # hold final position
frames[0].save(
    '/kaggle/working/game.gif',
    save_all=True, append_images=frames[1:],
    duration=600, loop=0
)
print(f'Saved: game.gif ({len(frames)} frames)')

print('\n── Download these before stopping the session ──')
print('  /kaggle/working/checkpoints/FINAL_best_agent.pt')
print('  /kaggle/working/training_curves.png')
print('  /kaggle/working/game.gif')
print('  /kaggle/working/logs/  (all 3 stage folders)')
print('\nACTION: Active Events (bottom-left) → Stop Session')